# Astro-QuFeX Playground

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
from datetime import datetime
from dataclasses import replace

import torch

from qmla.config import ConfigError, load_config
from qmla.engine import Trainer

ROOT = Path(".").resolve().parent
CONFIG = "experiments.toml"

RUN_ID = f"notebook_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

ROOT

WindowsPath('G:/MALTA/code/qmla')

In [2]:
config = load_config(ROOT / "configs" / CONFIG)
config.paths.runs_dir

WindowsPath('G:/MALTA/code/qmla/runs')

In [3]:
mode = config.run.model
run_dir = config.paths.runs_dir / RUN_ID

print("Configuration loaded successfully.")
print(f"Mode: {mode}")
print(f"Profile: {config.run.profile}")
print(f"Device: {config.training.device}")
print(f"Run directory: {run_dir}")


Configuration loaded successfully.
Mode: qufex
Profile: full64
Device: auto
Run directory: G:\MALTA\code\qmla\runs\notebook_20260916_094450


### Overrides
Simplify configuration parameters for quick prototyping

In [4]:
config = replace(
    config,
    training=replace(config.training, epochs=3)
)
config = replace(
    config,
    training=replace(config.training, device="cpu")
)

In [5]:
trainer = Trainer(config, run_dir, mode=mode)

In [10]:
for batch in trainer.train_loader:
    inputs, targets = batch
    inputs = inputs.to(trainer.device, non_blocking=True)
    if trainer.device.type == "cuda":
        inputs = inputs.contiguous(memory_format=trainer.torch.channels_last)
    targets = targets.to(trainer.device, non_blocking=True)
    break

inputs_shape = inputs.shape
print(inputs_shape)

torch.Size([8, 3, 64, 64])


In [ ]:
from torchview import draw_graph
from torchinfo import summary

summary(trainer.model, input_size=inputs_shape)

GalaxyClassifier(
  (encoder): Sequential(
    (0): ConvBlock(
      (block): Sequential(
        (0): Conv2d(3, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): Conv2d(4, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (4): BatchNorm2d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU()
        (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (1): ConvBlock(
      (block): Sequential(
        (0): Conv2d(4, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): Conv2d(8, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (4): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, tra

Layer (type:depth-idx)                   Output Shape              Param #
GalaxyClassifier                         [8, 3]                    --
├─Sequential: 1-1                        [8, 16, 2, 2]             --
│    └─ConvBlock: 2-1                    [8, 4, 32, 32]            --
│    │    └─Sequential: 3-1              [8, 4, 32, 32]            268
│    └─ConvBlock: 2-2                    [8, 8, 16, 16]            --
│    │    └─Sequential: 3-2              [8, 8, 16, 16]            896
│    └─ConvBlock: 2-3                    [8, 8, 8, 8]              --
│    │    └─Sequential: 3-3              [8, 8, 8, 8]              1,184
│    └─ConvBlock: 2-4                    [8, 8, 4, 4]              --
│    │    └─Sequential: 3-4              [8, 8, 4, 4]              1,184
│    └─ConvBlock: 2-5                    [8, 16, 2, 2]             --
│    │    └─Sequential: 3-5              [8, 16, 2, 2]             3,520
├─Sequential: 1-2                        [8, 16, 2, 2]             --
│   

In [14]:
config_direct = replace(
	config,
 	run = replace(config.run, model="direct_cnn")
)

trainer_cnn = Trainer(config_direct, run_dir, mode=mode)
summary(trainer.model, input_size=inputs_shape)

Layer (type:depth-idx)                   Output Shape              Param #
GalaxyClassifier                         [8, 3]                    --
├─Sequential: 1-1                        [8, 16, 2, 2]             --
│    └─ConvBlock: 2-1                    [8, 4, 32, 32]            --
│    │    └─Sequential: 3-1              [8, 4, 32, 32]            268
│    └─ConvBlock: 2-2                    [8, 8, 16, 16]            --
│    │    └─Sequential: 3-2              [8, 8, 16, 16]            896
│    └─ConvBlock: 2-3                    [8, 8, 8, 8]              --
│    │    └─Sequential: 3-3              [8, 8, 8, 8]              1,184
│    └─ConvBlock: 2-4                    [8, 8, 4, 4]              --
│    │    └─Sequential: 3-4              [8, 8, 4, 4]              1,184
│    └─ConvBlock: 2-5                    [8, 16, 2, 2]             --
│    │    └─Sequential: 3-5              [8, 16, 2, 2]             3,520
├─Sequential: 1-2                        [8, 16, 2, 2]             --
│   

## Train or Load

In [53]:
TRAIN = False
RUN_TO_LOAD = "20260914"

def find_run_dir(run_id: str) -> Path:
    run_dir = Path(run_id)
    if run_dir.is_dir():
        return run_dir
    
    run_dir = ROOT / run_id
    if run_dir.is_dir():
        return run_dir
    
    run_dir = ROOT / "runs" / run_id
    if run_dir.is_dir():
        return run_dir
    
    run_dir = ROOT / "runs" / f"notebook_{run_id}"
    if run_dir.is_dir():
        return run_dir
    
    run_id_to_lookfor = [f for f in ROOT.glob(f"runs/notebook_{run_id}*")]
    return run_id_to_lookfor[0]

runned_dir = find_run_dir(RUN_TO_LOAD)


In [54]:
runned_dir.stem

'notebook_20260914_181555'

In [55]:
if TRAIN:
	trainer.run()
else:
    checkpoint_dir = ROOT / "checkpoints" / runned_dir.stem
    checkpoint = torch.load(checkpoint_dir / "best.pt", trainer.config.training.device, weights_only=False)
    
